In [0]:
from pyspark.sql import functions as F, Window

bene = spark.table("claims.bronze.beneficiary")

def d(col):                      # CMS dates are yyyyMMdd strings
    return F.to_date(F.col(col), "yyyyMMdd")

latest = Window.partitionBy("DESYNPUF_ID").orderBy(F.col("_ingested_at").desc())

silver_bene = (bene
    .withColumn("_rn", F.row_number().over(latest))
    .filter("_rn = 1").drop("_rn")
    .select(
        F.col("DESYNPUF_ID").alias("beneficiary_id"),
        d("BENE_BIRTH_DT").alias("birth_date"),
        d("BENE_DEATH_DT").alias("death_date"),
        F.col("BENE_SEX_IDENT_CD").alias("sex_code"),
        F.col("BENE_RACE_CD").alias("race_code"),
        F.col("SP_STATE_CODE").alias("state_code"),
        F.col("BENE_COUNTY_CD").alias("county_code"))
    .withColumn("age_2010",
        F.floor(F.datediff(F.lit("2010-12-31"), F.col("birth_date")) / 365.25)))

silver_bene.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("claims.silver.beneficiary")

Unified claim header

In [0]:
def normalize(df, claim_type):
    return df.select(
        F.col("DESYNPUF_ID").alias("beneficiary_id"),
        F.col("CLM_ID").alias("claim_id"),
        F.lit(claim_type).alias("claim_type"),
        d("CLM_FROM_DT").alias("service_start"),
        d("CLM_THRU_DT").alias("service_end"),
        F.col("PRVDR_NUM").alias("provider_id"),
        F.col("CLM_PMT_AMT").cast("decimal(12,2)").alias("paid_amount"),
        F.col("_source_file"),
        F.col("_ingested_at"))

ip = normalize(spark.table("claims.bronze.inpatient"),  "inpatient")
op = normalize(spark.table("claims.bronze.outpatient"), "outpatient")

claim_header = (ip.unionByName(op)
    .dropDuplicates(["claim_id"])
    .withColumn("service_days", F.datediff("service_end", "service_start") + 1)
    .withColumn("service_month", F.date_trunc("month", "service_start")))

claim_header.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("claims.silver.claim_header")

Carrier service lines — the unpivot

In [0]:
carrier = spark.table("claims.bronze.carrier")
available = set(carrier.columns)

frames = []
for n in range(1, 14):
    needed = [f"HCPCS_CD_{n}", f"LINE_NCH_PMT_AMT_{n}"]
    if not all(c in available for c in needed):
        break                    # stop at the first missing position

    def opt(name, default=None):
        return F.col(name) if name in available else F.lit(default)

    frames.append(
        carrier.select(
            F.col("DESYNPUF_ID").alias("beneficiary_id"),
            F.col("CLM_ID").alias("claim_id"),
            F.lit(n).alias("line_number"),
            F.col(f"HCPCS_CD_{n}").alias("hcpcs_code"),
            opt(f"LINE_NCH_PMT_AMT_{n}").cast("decimal(12,2)").alias("line_paid"),
            opt(f"LINE_ALOWD_CHRG_AMT_{n}").cast("decimal(12,2)").alias("line_allowed"),
            opt(f"LINE_BENE_PTB_DDCTBL_AMT_{n}").cast("decimal(12,2)").alias("line_deduct"),
            opt(f"LINE_COINSRNC_AMT_{n}").cast("decimal(12,2)").alias("line_coins"),
            opt(f"LINE_ICD9_DGNS_CD_{n}").alias("line_dx"),
        ).filter(F.col("hcpcs_code").isNotNull()))

from functools import reduce
service_line = reduce(lambda a, b: a.unionByName(b), frames)

service_line = (service_line
    .withColumn("patient_responsibility",
        F.coalesce("line_deduct", F.lit(0)) + F.coalesce("line_coins", F.lit(0)))
    .withColumn("payment_variance",
        F.coalesce("line_allowed", F.lit(0)) - F.coalesce("line_paid", F.lit(0)))
    .withColumn("is_zero_pay",
        F.coalesce(F.col("line_paid"), F.lit(0)) == 0))

service_line.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("claims.silver.service_line")

print("service lines:", service_line.count())